In [10]:
from dotenv import load_dotenv
import os
import time
import json
import hashlib
from pathlib import Path
import pandas as pd
import psycopg2
from psycopg2.extras import execute_values
from datetime import datetime

load_dotenv()

DB_PARAMS = {
    "dbname": os.getenv("DB_NAME", "NYC"),
    "user": os.getenv("DB_USER", "postgres"),
    "password": os.getenv("DB_PASSWORD", "password"),
    "host": os.getenv("DB_HOST", "localhost"),
    "port": os.getenv("DB_PORT", "5432"),
}

csv_path = Path("NewYork_transportations.csv")
state_path = Path("pipeline_state.json")
WATCH_INTERVAL_SECONDS = 30
RUN_MODE = "batch"

In [11]:
def get_db_connection():
    try:
        conn = psycopg2.connect(**DB_PARAMS)
        print("🔗 Database connection successful.")
        return conn
    except Exception as e:
        print(f"❌ Error connecting to the database: {e}")
        raise


def load_csv(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path, parse_dates=["date"], infer_datetime_format=True)
    df = df.rename(columns={c: c.strip() for c in df.columns})
    print(f"Loaded {len(df)} rows from {path}")
    print(f"Columns: {list(df.columns)}")
    print(f"Transport types: {df['transport_type'].unique() if 'transport_type' in df.columns else []}")
    return df


def compute_file_hash(path: Path) -> str:
    hasher = hashlib.sha256()
    with path.open("rb") as f:
        hasher.update(f.read())
    return hasher.hexdigest()


def load_state() -> dict:
    if state_path.exists():
        return json.loads(state_path.read_text())
    return {}


def save_state(state: dict):
    state_path.write_text(json.dumps(state, indent=2))


def ensure_unique_indexes(cursor):
    cursor.execute(
        """
        DO $$
        BEGIN
            IF NOT EXISTS (
                SELECT 1 FROM pg_class c
                JOIN pg_namespace n ON n.oid = c.relnamespace
                WHERE c.relkind = 'i'
                  AND c.relname = 'daily_ridership_date_transport_type_key'
            ) THEN
                CREATE UNIQUE INDEX daily_ridership_date_transport_type_key
                ON daily_ridership (date, transport_type);
            END IF;

            IF NOT EXISTS (
                SELECT 1 FROM pg_class c
                JOIN pg_namespace n ON n.oid = c.relnamespace
                WHERE c.relkind = 'i'
                  AND c.relname = 'yearly_ridership_year_transport_type_key'
            ) THEN
                CREATE UNIQUE INDEX yearly_ridership_year_transport_type_key
                ON yearly_ridership (year, transport_type);
            END IF;

            IF NOT EXISTS (
                SELECT 1 FROM pg_class c
                JOIN pg_namespace n ON n.oid = c.relnamespace
                WHERE c.relkind = 'i'
                  AND c.relname = 'transport_types_transport_type_key'
            ) THEN
                CREATE UNIQUE INDEX transport_types_transport_type_key
                ON transport_types (transport_type);
            END IF;
        END$$
        """
    )


def insert_transport_types(cursor, df: pd.DataFrame):
    if "transport_type" not in df.columns:
        raise ValueError("CSV must contain a transport_type column.")

    unique_types = df["transport_type"].dropna().unique().tolist()
    cursor.executemany(
        "INSERT INTO transport_types (transport_type) VALUES (%s) ON CONFLICT DO NOTHING;",
        [(t,) for t in unique_types],
    )
    print(f"✓ Upserted {len(unique_types)} transport types")


def insert_daily_ridership(cursor, rows, truncate: bool = False):
    if truncate:
        cursor.execute("TRUNCATE TABLE daily_ridership;")
        print("✓ Cleared daily_ridership for full reload")

    if not rows:
        return 0

    insert_query = """
    INSERT INTO daily_ridership (date, ridership, transport_type, year)
    VALUES %s
    ON CONFLICT DO NOTHING;
    """
    execute_values(cursor, insert_query, rows, page_size=1000)
    return len(rows)


def refresh_yearly_aggregates(cursor, truncate: bool = False):
    if truncate:
        cursor.execute("TRUNCATE TABLE yearly_ridership;")
        print("✓ Cleared yearly_ridership for full reload")

    cursor.execute(
        """
        INSERT INTO yearly_ridership (year, transport_type, total_ridership)
        SELECT year, transport_type, SUM(ridership)
        FROM daily_ridership
        GROUP BY year, transport_type
        ON CONFLICT (year, transport_type) DO UPDATE
          SET total_ridership = EXCLUDED.total_ridership;
        """
    )


def verify_database(cursor):
    cursor.execute("SELECT COUNT(*) FROM transport_types;")
    count_types = cursor.fetchone()[0]

    cursor.execute("SELECT COUNT(*) FROM daily_ridership;")
    count_daily = cursor.fetchone()[0]

    cursor.execute("SELECT COUNT(*) FROM yearly_ridership;")
    count_yearly = cursor.fetchone()[0]

    print(f"\n✓ Transport Types: {count_types} records")
    print(f"✓ Daily Ridership: {count_daily} records")
    print(f"✓ Yearly Ridership: {count_yearly} records")

    cursor.execute(
        "SELECT year, transport_type, total_ridership FROM yearly_ridership ORDER BY year, transport_type LIMIT 10;"
    )
    print("\nSample yearly ridership data:")
    for year, transport_type, total in cursor.fetchall():
        print(f"  Year: {year}, Type: {transport_type}, Total: {total:,.2f}")


def process_csv():
    df = load_csv(csv_path)
    file_hash = compute_file_hash(csv_path)
    state = load_state()

    full_reload = False
    incremental_rows = []

    if not state:
        print("Starting first run: full load.")
        full_reload = True
    elif file_hash != state.get("file_hash"):
        if len(df) > state.get("row_count", 0):
            print("Detected new rows appended to CSV. Processing incremental load.")
            incremental_rows = df.iloc[state.get("row_count", 0) :].copy()
        else:
            print("Detected a file change that is not a simple append. Running a full reload.")
            full_reload = True
    else:
        print("No changes detected in CSV. Skipping load.")
        return

    conn = get_db_connection()
    cursor = conn.cursor()
    try:
        ensure_unique_indexes(cursor)
        insert_transport_types(cursor, df)

        if full_reload:
            daily_rows = [
                (
                    row["date"],
                    float(row["ridership"]),
                    row["transport_type"],
                    int(row["year"]),
                )
                for _, row in df.iterrows()
            ]
            inserted = insert_daily_ridership(cursor, daily_rows, truncate=True)
            refresh_yearly_aggregates(cursor, truncate=True)
            print(f"✓ Full reload inserted {inserted} daily ridership rows")
        else:
            daily_rows = [
                (
                    row["date"],
                    float(row["ridership"]),
                    row["transport_type"],
                    int(row["year"]),
                )
                for _, row in incremental_rows.iterrows()
            ]
            inserted = insert_daily_ridership(cursor, daily_rows)
            refresh_yearly_aggregates(cursor)
            print(f"✓ Incremental load inserted {inserted} new daily ridership rows")

        conn.commit()
        save_state({"file_hash": file_hash, "row_count": len(df), "last_run": datetime.utcnow().isoformat()})
        verify_database(cursor)
    except Exception as e:
        conn.rollback()
        print(f"✗ Error processing CSV: {e}")
        raise
    finally:
        cursor.close()
        conn.close()


def watch_csv(interval: int = WATCH_INTERVAL_SECONDS):
    print(f"Watching {csv_path} every {interval} seconds. Press interrupt to stop.")
    last_hash = None
    while True:
        if not csv_path.exists():
            print(f"Waiting for CSV file: {csv_path}")
        else:
            current_hash = compute_file_hash(csv_path)
            if current_hash != last_hash:
                process_csv()
                last_hash = current_hash
        time.sleep(interval)

In [12]:
if RUN_MODE == "watch":
    watch_csv()
else:
    process_csv()

Loaded 5376 rows from NewYork_transportations.csv
Columns: ['date', 'ridership', 'transport_type', 'year']
Transport types: ['subway' 'bus' 'ferry']
No changes detected in CSV. Skipping load.


C:\Users\kkr84\AppData\Local\Temp\ipykernel_34492\709492753.py:12: FutureWarning: The argument 'infer_datetime_format' is deprecated and will be removed in a future version. A strict version of it is now the default, see https://pandas.pydata.org/pdeps/0004-consistent-to-datetime-parsing.html. You can safely remove this argument.
  df = pd.read_csv(path, parse_dates=["date"], infer_datetime_format=True)
